# Authentic image dataset

Builds the **authentic (real) half** of the AI-GenBench dataset. The fake half ships prepackaged on
the Hub (`lrzpellegrini/AI-GenBench-fake_part`, see [datasets.ipynb](datasets.ipynb)); the real half
deliberately does not, it has to be reassembled from its origin datasets.

AI-GenBench defines the exact real set with file-ID lists in
`AI-GenBench/dataset_creation/resources/{train,validation}_real_file_ids.txt`. Reading those lists,
the authentic class is drawn from **three** sources:

| Source | What it is | Why it's in there |
|---|---|---|
| **COCO 2017** (train + val) | curated natural photos, captioned + segmented | content-paired reals for caption- and mask-conditioned fakes |
| **LAION-400M** (the ELSA_D3 subset) | web-scraped image/caption pairs | real counterpart to T2I fakes made from the same captions |
| **RAISE** | raw camera-original TIFFs | uncompressed photography — the hardest authentic case |

**NOTE:** The README also lists **ImageNet**, and a builder for it exists, but the shipped file-ID lists contain
zero ImageNet images. It is not part of the released benchmark, so this notebook ignores it.

### What this notebook does

The full real half is 180k images. We don't want that. Instead we pull a **small stratified sample**
straight over HTTP — COCO by image ID, LAION by scraped URL — never downloading an origin archive.
Every image goes through AI-GenBench's own preprocessing (`prepare_image`, `convert_to_jpeg=True`) so
the sample carries the **same JPEG history as the fake part**. That matters here more than usual: the
"Fake or JPEG?" confound means a mismatch in compression history between the two classes is enough to
manufacture discrimination that has nothing to do with synthesis.

Output is a JPEG folder plus a manifest, under `data/authentic/`.

## Setup

In [ ]:
import io
import json
import os
import random
import zipfile
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd()
os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / ".cache" / "huggingface"))

AIGENBENCH = PROJECT_ROOT / "AI-GenBench"
RESOURCES = AIGENBENCH / "dataset_creation" / "resources"
OUT_DIR = PROJECT_ROOT / "data" / "authentic"
IMG_DIR = OUT_DIR / "images"

SPLIT = "validation"  # "validation" (36k ids) or "train" (144k ids) — only the id list differs
N_PER_SOURCE = 34     # per-source sample size; 34 + 34 + N_RAISE = 72, matching the fake half
N_RAISE = 4           # RAISE TIFFs are ~20 MB each, so sample far fewer
SEED = 1234

# AI-GenBench constants, from dataset_creation/dataset_utils/common_utils.py
IMAGE_MIN_SIZE = 200
JPEG_QUALITY = 95
REAL_LABEL = 0

IMG_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
print("writing to", OUT_DIR)

In [ ]:
# The file-ID lists live in the AI-GenBench repo
if not AIGENBENCH.exists():
    !git clone --depth 1 https://github.com/MI-BioLab/AI-GenBench.git {AIGENBENCH}

file_ids = (RESOURCES / f"{SPLIT}_real_file_ids.txt").read_text().split()
by_source = Counter(fid.split("/")[0] for fid in file_ids)

print(f"{SPLIT} split: {len(file_ids):,} authentic images")
for source, n in by_source.most_common():
    print(f"  {source:<16} {n:>7,}  ({n / len(file_ids):.1%})")

## Fetch helpers

`prepare_image` below is a port of AI-GenBench's
[`dataset_utils/common_utils.py:prepare_image`](AI-GenBench/dataset_creation/dataset_utils/common_utils.py)
with `convert_to_jpeg=True`: strip XMP, flatten transparency onto white, convert to RGB, encode JPEG
at quality 95. It returns *bytes* rather than a `PIL.Image` so we can write exactly those bytes to
disk — re-saving a decoded image would silently put a second JPEG generation on every authentic file
and bias the very artifacts we're trying to measure.

Validation follows the same order AI-GenBench uses: `verify()` for structure, a real `load()` to
catch truncated scrapes, then the 200 px floor.

In [ ]:
import requests
from PIL import Image

SESSION = requests.Session()
SESSION.headers["User-Agent"] = "anchor-date-forensics/0.1 (thesis prototype)"


def fetch(url, timeout=20):
    """GET raw bytes. Returns (content, reason) — content is None on failure.

    Failures are expected here (LAION link rot), so we return the reason rather than raising, and
    tally the reasons in `harvest`: a systematic breakage and ordinary rot both show up as missing
    images, and only the breakdown tells them apart.
    """
    try:
        response = SESSION.get(url, timeout=timeout)
        response.raise_for_status()
        return response.content, "ok"
    except requests.HTTPError as error:
        return None, f"http {error.response.status_code}"
    except Exception as error:
        return None, type(error).__name__


def decode_and_validate(raw):
    """verify() -> reopen -> load() -> size floor. Returns the image, or None if it doesn't pass."""
    try:
        Image.open(io.BytesIO(raw)).verify()  # verify() leaves the handle unusable...
        image = Image.open(io.BytesIO(raw))   # ...so reopen before actually decoding
        image.load()
    except Exception:
        return None
    if min(image.size) < IMAGE_MIN_SIZE:
        return None
    return image


def prepare_image(image):
    """Port of AI-GenBench prepare_image(convert_to_jpeg=True). Returns encoded JPEG bytes."""
    image.info.pop("xmp", None)
    if image.mode != "RGB":
        if image.mode != "RGBA":
            image = image.convert("RGBA")
        background = Image.new("RGBA", image.size, (255, 255, 255))
        image = Image.alpha_composite(background, image).convert("RGB")
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=JPEG_QUALITY)
    return buffer.getvalue()


def harvest(candidates, n, origin, url_of, description_of=lambda candidate: ""):
    """Walk candidates until n images survive fetch + validation. Writes JPEGs, returns manifest rows.

    `candidates` are (file_id, payload) pairs and must already be shuffled: we stop at the first n
    that work, so the oversampling that covers link rot only stays unbiased if the order is random.

    Images already on disk are reused rather than refetched, so raising n on a later run only pulls
    the shortfall. That matters most for RAISE, where a single TIFF is most of the sample's bytes.
    """
    rows, outcomes = [], Counter()
    for file_id, payload in candidates:
        if len(rows) >= n:
            break
        path = IMG_DIR / (file_id.replace("/", "_") + ".jpg")
        if path.exists():
            with Image.open(path) as cached:
                width, height = cached.size
            outcomes["cached"] += 1
            rows.append({
                "file_id": file_id,
                "origin_dataset": origin,
                "label": REAL_LABEL,
                "generator": "",
                "description": description_of(payload),
                "width": width,
                "height": height,
                "path": str(path.relative_to(PROJECT_ROOT)),
            })
            continue
        raw, reason = fetch(url_of(payload))
        if raw is None:
            outcomes[reason] += 1
            continue
        image = decode_and_validate(raw)
        if image is None:
            outcomes["undecodable or under 200px"] += 1
            continue
        outcomes["ok"] += 1
        path.write_bytes(prepare_image(image))
        rows.append({
            "file_id": file_id,
            "origin_dataset": origin,
            "label": REAL_LABEL,
            "generator": "",
            "description": description_of(payload),
            "width": image.width,
            "height": image.height,
            "path": str(path.relative_to(PROJECT_ROOT)),
        })
    cached = outcomes["cached"]
    attempted = sum(outcomes.values()) - cached
    suffix = f" (+{cached} already on disk)" if cached else ""
    print(f"{origin:<16} kept {len(rows)}/{attempted} attempted{suffix}")
    for reason, count in outcomes.most_common():
        if reason not in ("ok", "cached"):
            print(f"  {reason:<28} {count}")
    return rows

## COCO 2017

COCO images are individually addressable on `images.cocodataset.org`, so we resolve the file IDs
directly instead of pulling the 19 GB train archive. Captions are left empty — they live in a 241 MB
annotations zip and the `description` field is metadata we don't need for the real class.

In [ ]:
coco_ids = [fid for fid in file_ids if fid.startswith("COCO2017_")]
random.shuffle(coco_ids)

COCO_DIRS = {"COCO2017_train": "train2017", "COCO2017_val": "val2017"}

# Go through the S3 endpoint, not the images.cocodataset.org vanity host: that host is a CNAME onto
# the same bucket but serves an *.s3.amazonaws.com certificate, so HTTPS fails hostname validation.
COCO_BASE = "https://s3.amazonaws.com/images.cocodataset.org"


def coco_url(file_id):
    prefix, image_id = file_id.split("/")
    return f"{COCO_BASE}/{COCO_DIRS[prefix]}/{int(image_id):012d}.jpg"


coco_rows = harvest(
    candidates=((fid, fid) for fid in coco_ids),
    n=N_PER_SOURCE,
    origin="COCO2017",
    url_of=coco_url,
)

## LAION-400M (ELSA_D3 subset)

AI-GenBench ships the scraped URLs alongside the file IDs, so no re-derivation from the LAION index is
needed. These are live web URLs from a 2021-era crawl and a good share of them are dead — that's the
link rot CLAUDE.md flags. `harvest` handles it by walking a shuffled candidate list until enough
images survive, so the survival rate printed below is the number to watch: if it's very low, widen
the candidate pool rather than lowering `N_PER_SOURCE`.

In [ ]:
laion_filelist = RESOURCES / f"{SPLIT}_laion400m_filelist.json"
if not laion_filelist.exists():
    with zipfile.ZipFile(laion_filelist.with_suffix(".zip")) as archive:
        archive.extractall(RESOURCES)

laion_entries = json.loads(laion_filelist.read_text())
print(f"{len(laion_entries):,} LAION URLs in the {SPLIT} filelist")

# The filelist already includes spare images beyond the ids actually used, precisely to absorb rot.
random.shuffle(laion_entries)

laion_rows = harvest(
    candidates=((f"LAION-400M/{entry['id']}", entry) for entry in laion_entries),
    n=N_PER_SOURCE,
    origin="LAION-400M",
    url_of=lambda entry: entry["url"],
    description_of=lambda entry: entry.get("description", ""),
)

## RAISE

RAISE is the one source with no open URL list: the CSV sits behind a confirmation page at
[loki.disi.unitn.it/RAISE](http://loki.disi.unitn.it/RAISE/confirm.php?package=all) ("Get the
images!" link). Drop `RAISE_all.csv` next to this notebook and the cell picks up from there;
otherwise it skips, and the rest of the notebook still works.

Worth the manual step: RAISE is the only authentic source here that is camera-original rather than
web-delivered, which makes it the closest available stand-in for genuinely digitized material — and
the sharpest test of whether a detector is keying on synthesis or on JPEG history. It is still not
archival scan material, which remains the gap this thesis has to fill separately.

In [ ]:
raise_csv = PROJECT_ROOT / "RAISE_urls.csv"
raise_rows = []

if not raise_csv.exists():
    print(f"{raise_csv.name} not found — skipping RAISE.")
    print("Get it from http://loki.disi.unitn.it/RAISE/confirm.php?package=all ('Get the images!')")
else:
    import pandas as pd

    # file_ids look like "RAISE/r0a2e85f0t"; the CSV keys the same ids in its File column.
    wanted = {fid.split("/")[1]: fid for fid in file_ids if fid.startswith("RAISE/")}
    catalog = pd.read_csv(raise_csv)
    matched = [
        (wanted[row.File], row.TIFF) for row in catalog.itertuples() if row.File in wanted
    ]
    random.shuffle(matched)
    print(f"{len(matched)} of {len(wanted)} {SPLIT}-split RAISE ids found in the CSV")

    # These are ~20 MB TIFFs, so N_RAISE stays well below N_PER_SOURCE.
    raise_rows = harvest(
        candidates=matched,
        n=N_RAISE,
        origin="RAISE",
        url_of=lambda url: url,
    )

## Manifest

One row per authentic image, with the columns from AI-GenBench's dataset schema that apply to the
real class (`label=0`, empty `generator`). Written as Parquet for downstream use and JSONL to keep it
diffable.

In [ ]:
import pandas as pd

manifest = pd.DataFrame(coco_rows + laion_rows + raise_rows)

manifest.to_parquet(OUT_DIR / f"authentic_{SPLIT}.parquet", index=False)
with open(OUT_DIR / f"authentic_{SPLIT}.jsonl", "w") as handle:
    for row in manifest.to_dict("records"):
        handle.write(json.dumps(row) + "\n")

total_mb = sum(p.stat().st_size for p in IMG_DIR.glob("*.jpg")) / 1e6
print(f"{len(manifest)} authentic images, {total_mb:.1f} MB on disk")
print(manifest["origin_dataset"].value_counts().to_string())
manifest.head()

In [ ]:
import matplotlib.pyplot as plt

sample = manifest.sample(min(6, len(manifest)), random_state=SEED)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    ax.imshow(Image.open(PROJECT_ROOT / row["path"]))
    ax.set_title(f'{row["origin_dataset"]}\n{row["width"]}×{row["height"]}', fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()

## Notes

- **This is a sample, not the benchmark.** Reproducing AI-GenBench's real half exactly means taking
  every id in the filelist and running `simple_dataset_generation.py`; what's here is a stratified
  peek for prototyping. The sampling is by source, not proportional — LAION is ~66% of the real
  ids and COCO ~33%, but we take equal counts.
- **Pairing is dropped.** AI-GenBench gives priority to *aligned* reals (the `paired_real_images`
  column — reals sharing a caption, mask, or inpainting source with a specific fake) so the two
  classes match on content. Sampling ids independently loses that. Anything comparing real vs. fake
  score distributions on this sample is confounded by content and needs the pairing restored first.
- **No ImageNet**, despite the README listing it — the shipped filelists contain none.
- **No archival scans.** RAISE is camera-raw, not digitized print or film. Measuring the false
  positive rate on genuinely digitized TIFFs is the thesis's central open question and needs a source
  outside this benchmark.